In [ ]:
import sys
import subprocess

required = ["torch", "transformers", "datasets", "scikit-learn"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])
print("Installed required packages.")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
model_name = "textattack/distilbert-base-uncased-MRPC"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

print(f"Loaded model: {model_name}")
print(f"Number of labels: {model.config.num_labels}")

In [ ]:
max_examples = 128
dataset = load_dataset("glue", "mrpc", split=f"validation[:{max_examples}]")

print("Dataset split: glue/mrpc validation")
print(f"Using first {len(dataset)} examples for fast evaluation")
print("Example row:")
print(dataset[0])

In [ ]:
batch_size = 32
labels = dataset["label"]
predictions = []
confidences = []
predicted_positive_prob = []

for start_idx in range(0, len(dataset), batch_size):
    batch = dataset[start_idx:start_idx + batch_size]
    inputs = tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(logits, dim=-1)

    predictions.extend(preds.cpu().tolist())
    confidences.extend(probs.max(dim=-1).values.cpu().tolist())
    predicted_positive_prob.extend(probs[:, 1].cpu().tolist())

print(f"Completed inference for {len(predictions)} examples.")

In [ ]:
accuracy = accuracy_score(labels, predictions)
precision, recall, f1, support_binary = precision_recall_fscore_support(labels, predictions, average="binary")
cm = confusion_matrix(labels, predictions)
per_class_precision, per_class_recall, per_class_f1, per_class_support = precision_recall_fscore_support(
    labels, predictions, labels=[0, 1], average=None, zero_division=0
)

print("Evaluation metrics on fixed 128-example subset:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print("Confusion matrix:")
print(cm)

print("Per-class metrics:")
label_map = {0: "not_paraphrase", 1: "paraphrase"}
for idx, label_id in enumerate([0, 1]):
    print(
        f"class={label_id} ({label_map[label_id]}), "
        f"precision={per_class_precision[idx]:.4f}, "
        f"recall={per_class_recall[idx]:.4f}, "
        f"f1={per_class_f1[idx]:.4f}, "
        f"support={per_class_support[idx]}"
    )

In [ ]:
false_positives = []
false_negatives = []

for i in range(len(dataset)):
    row = dataset[i]
    record = {
        "index": i,
        "true_label": labels[i],
        "pred_label": predictions[i],
        "confidence": confidences[i],
        "p_paraphrase": predicted_positive_prob[i],
        "sentence1": row["sentence1"],
        "sentence2": row["sentence2"],
    }
    if labels[i] == 0 and predictions[i] == 1:
        false_positives.append(record)
    elif labels[i] == 1 and predictions[i] == 0:
        false_negatives.append(record)

print(f"False positives: {len(false_positives)}")
print(f"False negatives: {len(false_negatives)}")

def show_errors(title, rows, limit=5):
    print(title)
    if not rows:
        print("None")
        return
    print("idx | true | pred | conf | p_paraphrase | sentence1 | sentence2")
    for r in rows[:limit]:
        s1 = r["sentence1"].replace("\n", " ")[:70]
        s2 = r["sentence2"].replace("\n", " ")[:70]
        print(
            f"{r['index']:>3} | {r['true_label']} | {r['pred_label']} | "
            f"{r['confidence']:.4f} | {r['p_paraphrase']:.4f} | {s1} | {s2}"
        )

show_errors("Compact false positive table (first 5)", false_positives, limit=5)
print("-" * 120)
show_errors("Compact false negative table (first 5)", false_negatives, limit=5)

In [ ]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("dataset_split=glue/mrpc validation[:128]")
print(f"device={device}")
print(f"num_examples={len(dataset)}")
print(f"accuracy={accuracy:.4f}")
print(f"precision={precision:.4f}")
print(f"recall={recall:.4f}")
print(f"f1={f1:.4f}")
print(f"confusion_matrix={cm.tolist()}")
print(f"support_not_paraphrase={int(per_class_support[0])}")
print(f"support_paraphrase={int(per_class_support[1])}")
print(f"false_positives={len(false_positives)}")
print(f"false_negatives={len(false_negatives)}")